# 03. 1D Convolutional Neural Network (CNN) Training

### Architecture:
- **Input**: Tokenized integer sequences.
- **Embedding**: Learned dense vector representations.
- **1D Convolutions**: Multi-scale n-gram filter kernels (sizes 3, 4, 5) to capture local phrases.
- **Global Max Pooling**: Extracts the most salient features across time.
- **Dropout & Dense Head**: Regularization and softmax class probabilities.

In [ ]:
import os
import sys
import torch
from torch.utils.data import DataLoader
import pandas as pd

sys.path.append(os.path.abspath('..'))
from src.preprocessing.nlp_pipeline import DatasetValidator, Vocabulary, LabelEncoder, clean_text
from src.models.cnn import TextCNN
from src.training.trainer import train_pytorch_model, TextDataset
from src.evaluation.metrics import evaluate_predictions

# Load and Split Data
df = pd.read_csv('../data/sample_sentiment_benchmark.csv')
train_df, val_df, test_df = DatasetValidator.prepare_splits(df, 'text', 'sentiment')

vocab = Vocabulary()
vocab.build_vocab(train_df['clean_text'].tolist())

le = LabelEncoder()
le.fit(train_df['sentiment'].tolist())

# Create Datasets
X_train = vocab.batch_to_sequences(train_df['clean_text'].tolist(), max_len=64)
y_train = le.transform(train_df['sentiment'].tolist())
X_val = vocab.batch_to_sequences(val_df['clean_text'].tolist(), max_len=64)
y_val = le.transform(val_df['sentiment'].tolist())

train_loader = DataLoader(TextDataset(X_train, y_train), batch_size=8, shuffle=True)
val_loader = DataLoader(TextDataset(X_val, y_val), batch_size=8)

# Initialize TextCNN
model = TextCNN(vocab_size=len(vocab), num_classes=len(le))
print(model)

In [ ]:
# Train with Early Stopping
train_results = train_pytorch_model(
    model, train_loader, val_loader, num_epochs=10, lr=1e-3
)
print(f"Training complete in {train_results['training_time_seconds']}s")

In [ ]:
# Evaluate on Test Set
X_test = vocab.batch_to_sequences(test_df['clean_text'].tolist(), max_len=64)
y_test = le.transform(test_df['sentiment'].tolist())

with torch.no_grad():
    logits = model(torch.tensor(X_test, dtype=torch.long))
    preds = torch.argmax(logits, dim=1).numpy()

metrics = evaluate_predictions(y_test, preds, target_names=le.classes_)
print("Accuracy:", metrics["accuracy"])
print("Macro F1:", metrics["macro_f1"])
print("Confusion Matrix:\n", metrics["confusion_matrix"])